# Querying ERA5 with SQL

The following query computes the monthly area-weighted global mean of temperature over a year of ERA5 data, running on AWS Lambda.

Start with imports, and setting the AWS region and datasource.

In [ ]:
import os

from zarr_datafusion_lambda import sql

# Where ERA5 and the workers are. Read once a kernel, at the first query.
os.environ["AWS_REGION"] = "us-east-1"

tables = {"era5": {"url": "s3://earthmover-icechunk-era5/icechunkV2", "group": "single/spatial"}}

This is the SQL query we'll run to compute the global mean. (You can change the time range here if you like.)

In [ ]:
query = """
SELECT date_trunc('month', CAST(valid_time AS TIMESTAMP))              AS month,
       sum(t2m * cos(radians(latitude))) / sum(cos(radians(latitude))) AS mean_t2m
FROM era5
WHERE valid_time >= TIMESTAMP '2020-01-01' AND valid_time < TIMESTAMP '2021-01-01'
GROUP BY 1
ORDER BY 1
"""

Create a SQL session for the source tables, running up to `concurrency` workers at once.

Note: 990 should fit comfortably in Lambda's concurrency limit of 1,000, but if the query fails with `TooManyRequestsException` then lower this number.

In [ ]:
era5 = sql.Session(tables, concurrency=990)

Now run the query! (This should only take a few seconds.)

In [ ]:
result = era5.sql(query, progress=True)
result.to_pandas()

Finally, show the run's stats, its estimated cost (by querying CloudWatch), and then a timeline showing the function invocations.

In [ ]:
print(result.stats())

In [ ]:
print(result.cost())

In [ ]:
result.timeline()